# Fine-tuning with Unsloth

based on the tutorial:
https://docs.unsloth.ai/basics/tutorial-how-to-finetune-llama-3-and-use-in-ollama



##### In this notebook, we fine-tune the model which achieved the best result in the previous fine-tuning

**mistral-7b-v0.3-bnb-4bit**


In [1]:
import os
import sys
import json
import time
import pandas as pd
from datasets import Dataset
sys.path.insert(1, '../')
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from common import  get_prompt_template, is_command_classification_correct
from unsloth import FastLanguageModel
from unsloth import apply_chat_template
from unsloth import standardize_sharegpt
import itertools, json, shutil, os
import wandb
max_seq_length = 2048
dtype = None
load_in_4bit = True # 4bit quantization to reduce memory usage

os.environ['UNSLOTH_RETURN_LOGITS'] = '1'

/home/danielhenel/Desktop/jarvis/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-06-17 23:49:48.424286: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750196988.438138  176205 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750196988.442278  176205 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750196988.453207  176205 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more 

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:

command_list = [
    "STOP",
    "START",
    "ENGINE_STOP",
    "ENGINE_START",
    "STOP_IN_PITLANE_TOGGLE_ON",
    "STOP_IN_PITLANE_TOGGLE_OFF",
    "SPEED_BY_RC_TOGGLE_ON",
    "SPEED_BY_RC_TOGGLE_OFF",
    "FLAG_BY_RC_TOGGLE_ON",
    "FLAG_BY_RC_TOGGLE_OFF",
    "INTO_PITLANE_TOGGLE_ON",
    "INTO_PITLANE_TOGGLE_OFF",
    "JOYSTICK_TOGGLE_ON",
    "JOYSTICK_TOGGLE_OFF",
    "SPEED_REQUEST",
    "GG_SCALE"
]

chat_template = """{SYSTEM}
USER: {INPUT}
ASSISTANT: {OUTPUT}"""


### Prepare train and test subsets

Dataset: Basic commands v2.0

Commands: START, STOP, START_ENGINE, STOP_ENGINE, CRANK_REQUEST

Train 80 - Test 20 split

In [3]:
def get_train_data(tokenizer):
    with open("../data/request_commands_train_dataset_v1.0.json", 'r') as file:
        raw_train_dataset = json.load(file)

    system_prompt = get_prompt_template(command_list=command_list, prompt_version=3)

    train_data = []
    for command_class in raw_train_dataset:
        for input, output in raw_train_dataset[command_class].items():
            if input and output:
                train_data.append({
                    "conversations": [
                        {"from": "user", "content": input},
                        {"from": "assistant", "content": output}
                    ]
                })

    train_data = Dataset.from_list(train_data)

    train_data = standardize_sharegpt(train_data)
    print(train_data[-85])

    train_data = apply_chat_template(
        train_data,
        tokenizer = tokenizer,
        chat_template = chat_template,
        default_system_message = system_prompt
    )

    return train_data


def get_test_data():
    with open("../data/request_commands_test_dataset_v1.0.json", 'r') as file:
        raw_test_dataset = json.load(file)
    test_data = []
    for command_class in raw_test_dataset:
        for input, output in raw_test_dataset[command_class].items():
            if input and output:
                test_data.append({"input": input, "output": output})
    return test_data


### Model, tokenizer and trainer configuration

##### We test various configurations of the following params:
rank(r)

learning rate (lr)

gradient_accumulation_steps (grad_steps) 

number of epochs (epoch)

lora_alpha (alpha)

##### Then we evaluate the accuracy of each model agains our test dataset
##### Finally, we select the model of best accuracy



In [4]:
def get_model_and_tokenizer(model_name: str, r: int, alpha: int) -> tuple:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name =  model_name,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r = r,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj",],
        lora_alpha = alpha,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
        use_rslora = False,
        loftq_config = None,
    )
    return (model, tokenizer)

def get_training_args(lr: float, epoch: int, grad_steps: int, output_dir: str, run_name: str) -> SFTConfig:
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = grad_steps,
        num_train_epochs = epoch,
        learning_rate = lr,
        output_dir=output_dir,
        logging_steps = 5,
        save_strategy="epoch",
        save_total_limit=1,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        report_to = "wandb",
        dataset_text_field = "text",
        dataset_num_proc = 2,
        packing = False,
        max_seq_length=2048,
        run_name=run_name,
        warmup_steps = 5,
        seed = 3407,
    )
    return args

### Training and evaluation

In [5]:
results = {}

def prompt_model(model, tokenizer, messages):
    FastLanguageModel.for_inference(model)

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    output_ids = model.generate(
    input_ids,
    max_new_tokens=128,
    pad_token_id=tokenizer.eos_token_id
    )
    generated_ids = output_ids[0][input_ids.shape[-1]:]
    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return generated_text

def benchmark_model(model, tokenizer):
    test_dataset = get_test_data()
    predictions = {
        'Classification Result': [],
        'Prompt processing time': []
    }

    probability_threshold = 0.80

    
    for i in range(len(test_dataset)):

        messages=[
            {"role": "system",  "content": get_prompt_template(command_list=command_list, prompt_version=3)},
            {"role": "user",  "content": test_dataset[i]['input']},]
        
        start_time = time.time() 
        response = prompt_model(model, tokenizer, messages)
        end_time = time.time()
        prompt_processing_time = end_time - start_time 

        classification_result = is_command_classification_correct(response, test_dataset[i]["output"].split("\n"), None, version=2)

        predictions['Classification Result'].append(classification_result[0])
        predictions['Prompt processing time'].append(prompt_processing_time)

    predictions = pd.DataFrame(predictions)
    accuracy = predictions[predictions['Classification Result'] == "CLASSIFICATION_CORRECT"].shape[0] / predictions.shape[0] 
    accuracy *= 100 # as percentage
    avg_processing_time = predictions['Prompt processing time'].mean()

    return (accuracy, avg_processing_time)

In [6]:
unsloth_4bit_models = [
    "./models/mistral-7b-v0.3-bnb-4bit"
]
learning_rates = [5e-5]
epochs = [2]
grad_acc_steps = [4]
r_values = [32]

param_grid = list(itertools.product(learning_rates, epochs, grad_acc_steps, r_values))

results = []
best_accuracy = -float("inf")
best_model_dir = None

for model_name in unsloth_4bit_models:
    for lr, epoch, grad_steps, r in param_grid:
        alpha = r
        run_name = f"{model_name}_lr{lr}_ep{epoch}_gs{grad_steps}_r{r}_alpha{alpha}"
        output_dir = f"outputs/{run_name}"
        print(f"\nRunning config for {model_name}: {run_name}")

        # Initialize WandB
        wandb.init(mode="offline", name=run_name, reinit=True)

        # Get model and tokenizer
        model, tokenizer = get_model_and_tokenizer(model_name, r, alpha)

        # Get train, test and validation data
        train_dataset = get_train_data(tokenizer) 

        # Train and evaluate
        trainer = SFTTrainer(
            model = model,
            tokenizer = tokenizer,
            train_dataset=train_dataset,
            args = get_training_args(lr, epoch, grad_steps, output_dir, run_name)
        )
        trainer.train()

        # Evaluate on the test dataset
        test_accuracy, avg_processing_time = benchmark_model(model, tokenizer)

        # Log metrics to WandB
        wandb.log({
            "eval_accuracy": test_accuracy,
            "learning_rate": lr,
            "epochs": epoch,
            "grad_acc_steps": grad_steps,
            "r": r,
            "lora_alpha": alpha,
            "avg_processing_time" : avg_processing_time
        })

        # Save to results
        results.append({
            "model_name": model_name,
            "lr": lr,
            "epochs": epoch,
            "grad_acc_steps": grad_steps,
            "r": r,
            "alpha": alpha,
            'eval_accuracy': test_accuracy,
            "output_dir": output_dir,
            "avg_processing_time" : avg_processing_time,
        })

        # Save best model checkpoint
        if test_accuracy > best_accuracy:
            print(f"New best model found at {output_dir} (eval_accuracy={test_accuracy:.4f})")
            best_model_dir = output_dir
            best_accuracy = test_accuracy
            if os.path.exists("best_model_checkpoint"):
                shutil.rmtree("best_model_checkpoint")
            shutil.copytree(output_dir, "best_model_checkpoint")

        wandb.finish()


Running config for ./models/mistral-7b-v0.3-bnb-4bit: ./models/mistral-7b-v0.3-bnb-4bit_lr5e-05_ep2_gs4_r32_alpha32


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


==((====))==  Unsloth 2025.4.4: Fast Mistral patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 3060 Laptop GPU. Num GPUs = 1. Max memory: 5.799 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.4.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
Unsloth: Standardizing formats (num_proc=12): 100%|██████████| 1308/1308 [00:00<00:00, 5905.80 examples/s]
Unsloth: We automatically added an EOS token to stop endless generations.


{'conversations': [{'content': 'raise gg scale setting to zero point fifty five', 'role': 'user'}, {'content': 'GG_SCALE\n0.55\nINCREASE', 'role': 'assistant'}]}


...

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,1.472800
10,0.852000
15,0.230600
20,0.105200
25,0.095300
30,0.092800
35,0.083700
40,0.079900
45,0.079400
50,0.073200


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
wandb: ERROR The nbformat package was not found. It is required to save notebook history.


New best model found at outputs/./models/mistral-7b-v0.3-bnb-4bit_lr5e-05_ep2_gs4_r32_alpha32 (eval_accuracy=97.6261)


avg_processing_time,▁
epochs,▁
eval_accuracy,▁
grad_acc_steps,▁
learning_rate,▁
lora_alpha,▁
r,▁
train/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇███
train/global_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇██
train/grad_norm,█▃▂▁▁▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▇████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁


### Results

In [7]:
results.sort(key=lambda x: x['eval_accuracy'], reverse=True)

for result in results:
    print("Result:")
    for key, value in result.items():
        print(f"  {key}: {value}")
    print("-" * 40)

Result:
  model_name: ./models/mistral-7b-v0.3-bnb-4bit
  lr: 5e-05
  epochs: 2
  grad_acc_steps: 4
  r: 32
  alpha: 32
  eval_accuracy: 97.62611275964392
  output_dir: outputs/./models/mistral-7b-v0.3-bnb-4bit_lr5e-05_ep2_gs4_r32_alpha32
  avg_processing_time: 1.387055191866368
----------------------------------------
